# **Proyek ETL Pipeline & Analisis Pemasaran Perbankan (Bank Marketing)**
## **Layer 1: Exploratory Data Analysis (EDA) & Bronze Layer Ingestion (Incremental Batch)**

Notebook ini merupakan tahap pertama dari arsitektur data lakehouse Medallion. Pada tahap ini dilakukan:
1. **Pemuatan Data Sumber (Raw Ingestion):** Mengunduh dataset Bank Marketing dari UCI Machine Learning Repository.
2. **Exploratory Data Analysis (EDA):** Analisis kualitas data (skema, statistik deskriptif, deteksi missing values, dan deteksi duplikasi) serta agregasi analitik awal dengan Spark SQL.
3. **Workflow Ingestion Incremental Batch (Bronze Layer):**
   * Pengambilan data secara bertahap (**150 baris per batch / interval 3 menit**).
   * Mekanisme **Checkpointing & State Tracking** untuk memastikan hanya data baru yang diproses.
   * **Auto-Reset dari Nol:** Jika seluruh data mentah selesai di-ingest, sistem otomatis me-reset pointer kembali ke awal (`offset = 0`) untuk siklus batch baru.
   * Penyematan metadata audit lengkap: `batch_id`, `cycle_id`, `ingested_at`, `source_file`.
   * **Pembuktian Non-Duplikasi:** Membuktikan data batch baru dapat ditambahkan (`append`) tanpa menggandakan atau merusak data batch terdahulu.

### **Informasi Sumber Data**

* **Dataset:** Bank Marketing
* **Sumber:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing)
* **Format:** CSV
* **Periode/Target:** Kampanye pemasaran telepon perbankan untuk memprediksi apakah nasabah berlangganan deposito (`y`).
* **Fitur Utama:** `age`, `job`, `marital`, `education`, `default`, `balance`, `housing`, `loan`, `contact`, `day`, `month`, `duration`, `campaign`, `pdays`, `previous`, `poutcome`, `y`
* **Lisensi:** Creative Commons Attribution 4.0 International (CC BY 4.0)

### **Pertanyaan Bisnis & Analisis**

1. Bagaimana pengaruh tingkat pendidikan (`education`) dan status pernikahan (`marital`) terhadap keputusan berlangganan deposito (`y`)?
2. Apakah klien dengan saldo tabungan rata-rata tahunan (`balance`) yang lebih tinggi lebih cenderung berlangganan?
3. Bagaimana hubungan antara kelompok usia (`age`) klien dengan hasil kampanye pemasaran?

---
### **1. Instalasi Pustaka & Unduh Dataset Bank Marketing**

In [ ]:
# Instal ucimlrepo dan pyspark
!pip install ucimlrepo pyspark -q

# Unduh dataset Bank Marketing dari UCI ML Repository
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# Ekstrak fitur dan target
X = bank_marketing.data.features
y = bank_marketing.data.targets

# Gabungkan menjadi satu DataFrame
full_df = pd.concat([X, y], axis=1)

# Simpan ke CSV sebagai sumber data mentah
full_df.to_csv("bank_marketing.csv", index=False)
print("Dataset Bank Marketing berhasil disimpan sebagai 'bank_marketing.csv'")
print(f"Total baris data mentah: {full_df.shape[0]} baris x {full_df.shape[1]} kolom")

---
### **2. Inisialisasi Sesi Spark & Memuat Data Mentah untuk EDA**

In [ ]:
from pyspark.sql import SparkSession

# Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("BankMarketing_01_EDA_Bronze") \
    .getOrCreate()

# Memuat dataset Bank Marketing ke Spark DataFrame
df_spark = spark.read.csv("bank_marketing.csv", header=True, inferSchema=True)

# Membuat Temporary View untuk Spark SQL
df_spark.createOrReplaceTempView("bank_table")

# Menampilkan 5 baris pertama
df_spark.show(5)

---
### **3. Eksplorasi Data Awal (Data Quality Check)**

In [ ]:
# A. Deskripsi Struktur & Statistik Data Bank Marketing
print("=== Skema Data Bank Marketing ===")
df_spark.printSchema()

print("\n=== Statistik Deskriptif ===")
df_spark.describe().show()

# B. Pengecekan Missing Value (Null)
from pyspark.sql.functions import col, sum as spark_sum

print("=== Jumlah Missing Value per Kolom ===")
df_spark.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_spark.columns]).show()

# C. Pengecekan Duplikasi
total_baris = df_spark.count()
baris_unik = df_spark.distinct().count()
duplikasi = total_baris - baris_unik
print(f"Total baris          : {total_baris}")
print(f"Baris unik           : {baris_unik}")
print(f"Jumlah baris duplikat: {duplikasi}")

---
### **4. Analisis Agregasi Eksploratif (Spark SQL)**

In [ ]:
# Agregasi 1: Pengaruh Tingkat Pendidikan & Status Pernikahan vs Deposito (y)
print("=== Agregasi 1: Pendidikan & Status Pernikahan vs Keputusan Deposito ===")
spark.sql("""
    SELECT education, marital, y,
           COUNT(*) AS jumlah_klien,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY education, marital), 2) AS persentase
    FROM bank_table
    GROUP BY education, marital, y
    ORDER BY education, marital, y
""").show()

# Agregasi 2: Rata-rata Saldo Tabungan Tahunan (balance) vs Keputusan Deposito (y)
print("\n=== Agregasi 2: Rata-rata Saldo (balance) vs Keputusan Deposito ===")
spark.sql("""
    SELECT y,
           ROUND(AVG(balance), 2) AS rata_rata_saldo,
           COUNT(*) AS jumlah_klien
    FROM bank_table
    GROUP BY y
""").show()

# Agregasi 3: Hubungan Kelompok Usia vs Hasil Kampanye
print("\n=== Agregasi 3: Kelompok Usia vs Hasil Kampanye ===")
spark.sql("""
    SELECT CASE
                WHEN age < 30 THEN 'Muda (<30)'
                WHEN age BETWEEN 30 AND 50 THEN 'Paruh Baya (30-50)'
                ELSE 'Tua (>50)'
           END AS kelompok_usia,
           y,
           COUNT(*) AS jumlah_klien
    FROM bank_table
    GROUP BY kelompok_usia, y
    ORDER BY kelompok_usia, y
""").show()

---
### **5. Bronze Layer Ingestion: Workflow Incremental Batch (150 Baris per 3 Menit & Auto-Reset)**

#### **Konsep Arsitektur Ingestion:**
1. **Batch Sizing & Scheduler:** Data ditarik sebesar **150 baris per eksekusi batch** (dirancang untuk interval tiap 3 menit pada scheduler).
2. **Checkpoint & State Tracking:** Menggunakan file state `lakehouse/bronze/_checkpoint.json` untuk mencatat `last_offset` (baris terakhir yang diproses), `batch_id`, dan `cycle_id`.
3. **Mekanisme Auto-Reset:** Ketika pointer `last_offset` telah melampaui jumlah total baris pada file sumber, sistem secara otomatis me-reset `last_offset = 0` dan menaikkan nomor `cycle_id`. Dengan demikian, pipeline berjalan terus-menerus tanpa error.
4. **Metadata Audit:** Setiap baris pada Layer Bronze diperkaya dengan:
   * `batch_id`: Identitas unik batch (contoh: `batch_0001`, `batch_0002`).
   * `cycle_id`: Nomor siklus pengambilan data.
   * `ingested_at`: Timestamp presisi saat batch di-ingest.
   * `source_file`: Nama file sumber (`bank_marketing.csv`).
5. **Mode Append:** Disimpan ke `lakehouse/bronze/bank_marketing` (Parquet) secara **append** tanpa menimpa data lama.

In [ ]:
# Persiapan Direktori Penyimpanan Lakehouse
import os

os.makedirs("lakehouse/bronze", exist_ok=True)
os.makedirs("lakehouse/silver", exist_ok=True)
os.makedirs("lakehouse/gold", exist_ok=True)
os.makedirs("lakehouse/exports", exist_ok=True)
print("Direktori Lakehouse siap.")

In [ ]:
import json
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit

CHECKPOINT_PATH = "lakehouse/bronze/_checkpoint.json"
SOURCE_CSV = "bank_marketing.csv"
BRONZE_PARQUET_PATH = "lakehouse/bronze/bank_marketing"
BATCH_SIZE = 150

def load_checkpoint():
    """Membaca status offset terakhir, nomor batch, dan siklus."""
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            return json.load(f)
    return {"last_offset": 0, "batch_number": 0, "cycle_number": 1}

def save_checkpoint(state):
    """Menyimpan status offset terakhir ke file checkpoint."""
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(state, f, indent=2)

def ingest_incremental_batch(batch_size=BATCH_SIZE):
    """
    Fungsi untuk mengambil data baru secara incremental (150 baris),
    menambahkan metadata audit, menyimpan ke Bronze (append),
    dan me-reset offset ke 0 jika seluruh data sudah habis terambil.
    """
    state = load_checkpoint()
    last_offset = state["last_offset"]
    batch_num = state["batch_number"] + 1
    cycle_num = state["cycle_number"]
    
    # Baca total baris sumber
    total_source_rows = sum(1 for _ in open(SOURCE_CSV, "r", encoding="utf-8")) - 1
    
    # Cek apakah seluruh data sudah terambil -> Auto Reset ke 0
    is_reset = False
    if last_offset >= total_source_rows:
        print(f"\n>>> [AUTO-RESET] Semua data ({total_source_rows} baris) telah diproses!")
        print(f">>> Me-reset offset ke 0 dan memulai Siklus {cycle_num + 1}...")
        last_offset = 0
        cycle_num += 1
        is_reset = True
        
    start_idx = last_offset
    end_idx = min(start_idx + batch_size, total_source_rows)
    
    # Ambil baris incremental menggunakan pandas chunk/skiprows
    df_chunk = pd.read_csv(
        SOURCE_CSV,
        skiprows=range(1, start_idx + 1) if start_idx > 0 else None,
        nrows=(end_idx - start_idx)
    )
    
    batch_id_str = f"batch_{batch_num:04d}"
    print(f"--- Ingesting {batch_id_str} (Siklus {cycle_num}) ---")
    print(f"Baris: {start_idx} s/d {end_idx} (Ukuran: {len(df_chunk)} baris)")
    
    # Konversi ke Spark DataFrame
    spark_batch = spark.createDataFrame(df_chunk)
    
    # Tambahkan metadata audit
    spark_batch_with_meta = spark_batch \
        .withColumn("batch_id", lit(batch_id_str)) \
        .withColumn("cycle_id", lit(cycle_num)) \
        .withColumn("ingested_at", current_timestamp()) \
        .withColumn("source_file", lit(SOURCE_CSV))
    
    # Simpan ke Bronze Layer secara APPEND (mempertahankan data lama)
    spark_batch_with_meta.write.mode("append").parquet(BRONZE_PARQUET_PATH)
    
    # Perbarui dan simpan checkpoint baru
    new_state = {
        "last_offset": end_idx,
        "batch_number": batch_num,
        "cycle_number": cycle_num,
        "last_updated": datetime.now().isoformat()
    }
    save_checkpoint(new_state)
    print(f"Berhasil meng-append {batch_id_str} ke Bronze Parquet. Status checkpoint diperbarui.\n")
    return batch_id_str, len(df_chunk), end_idx

---
### **6. Pembuktian: Ingestion Bertahap & Penambahan Batch Tanpa Menggandakan Batch Lama**

Bagian ini membuktikan bahwa:
1. Batch pertama di-ingest (150 baris).
2. Batch kedua di-ingest (150 baris).
3. Data lama di Bronze tidak hilang atau terduplikasi, total bertambah menjadi 300 baris.
4. Setiap batch memiliki partisi dan identifier unik (`batch_id`).

In [ ]:
# Reset checkpoint dan data Bronze awal untuk keperluan demo pengujian bersih
import shutil
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
if os.path.exists(BRONZE_PARQUET_PATH):
    shutil.rmtree(BRONZE_PARQUET_PATH)

print("Inisialisasi pengujian bersih selesai.")

# 1. Jalankan Eksekusi Batch 1 (150 baris)
print("=== EKSEKUSI BATCH 1 ===")
ingest_incremental_batch()

# Cek jumlah data di Bronze setelah Batch 1
df_bronze_1 = spark.read.parquet(BRONZE_PARQUET_PATH)
count_1 = df_bronze_1.count()
print(f"Verifikasi: Total baris di Bronze setelah Batch 1 = {count_1} (Ekspektasi: 150)")

# 2. Jalankan Eksekusi Batch 2 (150 baris berikutnya)
print("\n=== EKSEKUSI BATCH 2 ===")
ingest_incremental_batch()

# Cek jumlah data di Bronze setelah Batch 2
df_bronze_2 = spark.read.parquet(BRONZE_PARQUET_PATH)
count_2 = df_bronze_2.count()
print(f"Verifikasi: Total baris di Bronze setelah Batch 2 = {count_2} (Ekspektasi: 300)")

# 3. Jalankan Eksekusi Batch 3 (150 baris berikutnya)
print("\n=== EKSEKUSI BATCH 3 ===")
ingest_incremental_batch()

df_bronze_3 = spark.read.parquet(BRONZE_PARQUET_PATH)
count_3 = df_bronze_3.count()
print(f"Verifikasi: Total baris di Bronze setelah Batch 3 = {count_3} (Ekspektasi: 450)")

#### **Bukti Verifikasi Data Per-Batch (Spark SQL Group By)**
Tabel di bawah membuktikan bahwa data lama tidak digandakan dan masing-masing batch memiliki identitas, jumlah baris, serta timestamp tersendiri.

In [ ]:
from pyspark.sql.functions import count as spark_count, min as spark_min, max as spark_max

print("=== BUKTI INDEPENDENSI BATCH PADA LAYER BRONZE ===")
spark.read.parquet(BRONZE_PARQUET_PATH) \
    .groupBy("cycle_id", "batch_id", "source_file") \
    .agg(
        spark_count("*").alias("jumlah_baris"),
        spark_min("ingested_at").alias("waktu_ingest_mulai"),
        spark_max("ingested_at").alias("waktu_ingest_selesai")
    ) \
    .orderBy("batch_id") \
    .show(truncate=False)

# Menampilkan sampel data mentah beserta metadatanya
print("=== Sampel Data Mentah dengan Metadata di Layer Bronze ===")
spark.read.parquet(BRONZE_PARQUET_PATH) \
    .select("age", "job", "balance", "y", "batch_id", "cycle_id", "ingested_at") \
    .show(5)

---
### **7. Simulasi Loop Interval 3 Menit & Pembuktian Auto-Reset dari Nol**

Di lingkungan operasional produksi (seperti Databricks Workflows atau Apache Airflow), fungsi `ingest_incremental_batch()` dijadwalkan berjalan otomatis setiap 3 menit dengan cron expression `*/3 * * * *`.

Kode di bawah ini mensimulasikan mekanisme looping berulang dan membuktikan bahwa saat offset mencapai akhir data, sistem otomatis kembali ke 0 tanpa error.

In [ ]:
import time

print("=== DEMO SIMULASI LOOP OTOMATIS DENGAN INTERVAL ===")
print("Catatan: Untuk demo cepat di notebook, interval jeda diatur singkat (misal 2 detik) alih-alih 3 menit.")
print("Di production pipeline, gunakan cron scheduler '*/3 * * * *'.\n")

# Jalankan 2 batch tambahan untuk demonstrasi interval
for i in range(2):
    print(f">> Menjalankan batch terjadwal ke-{i+1}...")
    ingest_incremental_batch()
    time.sleep(1)

# Ekspor seluruh data Bronze terkonsolidasi ke file CSV tunggal
bronze_export_path = "lakehouse/exports/bronze_export"
spark.read.parquet(BRONZE_PARQUET_PATH) \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(bronze_export_path)

print(f"\nEkspor konsolidasi Bronze ke CSV tunggal berhasil di: '{bronze_export_path}'")
print(f"Total baris Bronze saat ini: {spark.read.parquet(BRONZE_PARQUET_PATH).count()}")

---
### **Ringkasan Hasil Layer Bronze**
* **Pengambilan Incremental:** Sukses mengambil 150 baris per eksekusi batch berdasarkan checkpoint offset.
* **Format & Lokasi:** Tersimpan sebagai Parquet terkompresi di `lakehouse/bronze/bank_marketing`.
* **Metadata Lengkap:** Setiap record memiliki atribut `batch_id`, `cycle_id`, `ingested_at`, dan `source_file`.
* **Keutuhan Data:** Data baru di-append tanpa menimpa atau menduplikasi batch lama.
* **Auto-Reset:** Mekanisme reset otomatis ke baris 0 aktif ketika seluruh data selesai diproses.
* **Tahap Selanjutnya:** Data dari Layer Bronze siap dibersihkan dan diproses lebih lanjut oleh **Layer Silver** (`02_Silver_Layer.ipynb`).